In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from glob import glob
from tqdm import tqdm
from onnx2torch import convert
from src.service.fragment.net import Net
from src.utilities.load_dataset import load_dataset
from src.utilities.get_net_score import get_score_net
from src.utilities.get_macs import get_macs_params_onnx
from src.utilities.get_data_score import get_data_score
from skl2onnx.helpers.onnx_helper import load_onnx_model
from src.utilities.get_accuracy import accuracy_score_net
from src.utilities.visual import draw_stitchNet_fromTuples
from src.utilities.evaluate_model import eval_original_model
from src.utilities.dataloader_generator import generate_dataloader

In [3]:
data_score = get_data_score(16)

batch_size = 16
dataset_train = load_dataset()
dataset_val = load_dataset("test")
dataloaders = dict(
    train=generate_dataloader(dataset_train, batch_size=batch_size),
    val=generate_dataloader(dataset_val, batch_size=batch_size)
)
os.makedirs('../_results_with_finetune/original', exist_ok=True)

In [4]:
# evaluate accuracy of the original networks before finetuning


modelnames = sorted(glob('../_results_with_finetune/finetune/*/*.onnx'))
for i, modelname in tqdm(enumerate(modelnames), position=0, leave=True):
    namewithoutext = modelname.split("/")[2]
    if namewithoutext == "densenet121":
        continue
    # if os.path.exists(f'_results_with_finetune/original/{namewithoutext}.txt'):
    #     continue
    model_onnx1 = load_onnx_model(modelname)
    macs, params = get_macs_params_onnx(model_onnx1)
    torch_model_1 = convert(model_onnx1)
    valacc, trainacc = eval_original_model(torch_model_1, dataloaders)
    
    # get score
    fragmentFiles = sorted(glob(f'../_results_with_finetune/fragments/net{i:03}/*.onnx'))
    onnxFragments = []
    js = []
    for j,fragmentFile in enumerate(fragmentFiles):
        onnxFragment = load_onnx_model(fragmentFile)
        onnxFragments.append(onnxFragment)
        js.append((i,j))
    net1 = Net(onnxFragments, i)
    score = get_score_net(net1, data_score)
    
    accuracy = accuracy_score_net(Net([model_onnx1]), dataset_val, bs=256)
    print('ACC:',accuracy, valacc, trainacc)
    
    with open(f'../_results_with_finetune/original/{namewithoutext}.txt', 'w') as f:
        f.write(f'{valacc},{trainacc},{macs},{params},{score},"{tuple(js)}"\n')
        
    draw_stitchNet_fromTuples(js, name=f'../_results_with_finetune/original/{namewithoutext}')
        # f.write(f'{",".join([f"{x:.4}" for x in [valacc,trainacc,macs,params,score]])}\n')
    # break

0it [00:00, ?it/s]

Node Init Time Elapsed 0.0003979206085205078
Tensor Init Time Elapsed 0.19423913955688477
IO Tensor Init Time Elapsed 5.4836273193359375e-05
Constant Search Time Elapsed 1.4066696166992188e-05
Update Nodes Tensors  Time Elapsed 4.863739013671875e-05
{'Conv': 2.09808349609375e-05, 'Relu': 1.71661376953125e-05, 'MaxPool': 1.2636184692382812e-05, 'AveragePool': 3.337860107421875e-06, 'Flatten': 1.4781951904296875e-05, 'Gemm': 8.344650268554688e-06}


100%|█████████████████████████████████████████████████████| 162/162 [00:05<00:00, 30.32it/s]
7it [00:04,  1.50it/s]
100%|███████████████████████████████████████████████████████| 40/40 [00:20<00:00,  1.97it/s]
1it [00:53, 53.14s/it]

ACC: 3.5175879396984926 0.0 0.0
Node Init Time Elapsed 0.0013854503631591797
Tensor Init Time Elapsed 0.002294301986694336
IO Tensor Init Time Elapsed 0.0002484321594238281
Constant Search Time Elapsed 4.0531158447265625e-05
Update Nodes Tensors  Time Elapsed 0.0005002021789550781
{'Conv': 0.000148773193359375, 'HardSwish': 3.838539123535156e-05, 'Relu': 2.765655517578125e-05, 'GlobalAveragePool': 1.9311904907226562e-05, 'HardSigmoid': 1.71661376953125e-05, 'Mul': 3.4809112548828125e-05, 'Add': 2.2649765014648438e-05, 'Flatten': 8.821487426757812e-06, 'Gemm': 5.9604644775390625e-06}


100%|█████████████████████████████████████████████████████| 162/162 [00:05<00:00, 31.32it/s]
12it [00:01, 11.51it/s]
100%|███████████████████████████████████████████████████████| 40/40 [00:23<00:00,  1.69it/s]
3it [01:43, 32.42s/it]

ACC: 3.28643216080402 0.0 0.0
Node Init Time Elapsed 0.0013217926025390625
Tensor Init Time Elapsed 0.03096938133239746
IO Tensor Init Time Elapsed 0.0003299713134765625
Constant Search Time Elapsed 4.0531158447265625e-05
Update Nodes Tensors  Time Elapsed 0.0005099773406982422
{'Conv': 0.0001552104949951172, 'Relu': 9.322166442871094e-05, 'MaxPool': 7.867813110351562e-06, 'Add': 6.556510925292969e-05, 'GlobalAveragePool': 3.0994415283203125e-06, 'Flatten': 1.0251998901367188e-05, 'Gemm': 4.291534423828125e-06}


100%|█████████████████████████████████████████████████████| 162/162 [00:11<00:00, 13.98it/s]
5it [00:03,  1.53it/s]
100%|███████████████████████████████████████████████████████| 40/40 [00:45<00:00,  1.13s/it]
4it [03:29, 57.80s/it]

ACC: 3.6807112485504447 0.0 0.0
Node Init Time Elapsed 0.0004432201385498047
Tensor Init Time Elapsed 0.38056135177612305
IO Tensor Init Time Elapsed 0.0001068115234375
Constant Search Time Elapsed 1.9550323486328125e-05
Update Nodes Tensors  Time Elapsed 9.322166442871094e-05
{'Conv': 4.315376281738281e-05, 'Relu': 3.218650817871094e-05, 'MaxPool': 1.6450881958007812e-05, 'AveragePool': 3.814697265625e-06, 'Flatten': 1.7404556274414062e-05, 'Gemm': 8.106231689453125e-06}


100%|█████████████████████████████████████████████████████| 162/162 [00:18<00:00,  8.89it/s]
15it [00:19,  1.27s/it]
  0%|                                                                | 0/40 [00:00<?, ?it/s]2024-04-15 15:36:47.420278565 [E:onnxruntime:, sequential_executor.cc:514 ExecuteKernel] Non-zero status code returned while running FusedConv node. Name:'/features/features.2/Conv' Status Message: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:376 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*, bool, onnxruntime::WaitNotificationFn) Failed to allocate memory for requested buffer of size 3288334336


4it [05:22, 80.53s/it]


RuntimeException: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running FusedConv node. Name:'/features/features.2/Conv' Status Message: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:376 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*, bool, onnxruntime::WaitNotificationFn) Failed to allocate memory for requested buffer of size 3288334336
